# 03 — Data Cleaning & Integration (Shared Pipeline)

**Project:** Indonesia Financial Inclusion & Digital Finance Analytics  
**Owners:** Person 1 + Person 2 (joint review)  
**Input:** World Bank Global Findex Indonesia 2024, official `IDN_2024_FINDEX_v02_M_CSV.zip`  
**Runtime:** Google Colab or local Jupyter / Python 3.

### What this notebook does
1. Import and verify the ORIGINAL World Bank ZIP by **SHA-256**, not by shape alone.
2. Validate schema, survey routing, answer codes, respondent uniqueness, and positive survey weights.
3. Harmonize demographic labels and missing-value treatment across Notebooks **02** and **04**.
4. Create privacy-protected **respondent-level clean data** (local/private only).
5. Compute reproducible **weighted national/segment KPI** and **financial-barrier aggregates** for subsequent Looker Studio work.
6. Save a quality-check report, integrated dictionary and manifest.

### Critical correction from Notebook 02
The previously committed Notebook 02 contains an **incorrect mapping** for 2024 `fin11d`, `fin11e`, `fin11f`. The **2024** World Bank Indonesia codebook confirms:
- `fin11d`: **not enough money**.
- `fin11e`: **another family member has an account**.
- `fin11f`: **lack of trust**.

These meanings differ from some **2021** codebooks, so do not reuse 2021 labels. See the [official 2024 Indonesia dictionary](https://microdata.worldbank.org/catalog/7917/data-dictionary/F1).

**Data governance:** The original ZIP and respondent-level output are **not** to be pushed to this public GitHub repository. `.gitignore` excludes CSV/ZIP files. Keep respondent-level files local or in a restricted private Drive folder.

**Run all cells in order.** No external database or Google Cloud billing is required.


## 1. Set up Python environment, locate repository and load original ZIP

In **Colab**, opening this notebook from GitHub automatically clones the repository into the Colab runtime (if needed), then asks you to upload the original ZIP. On a **local machine**, clone the repository, put the original ZIP inside `data/raw/`, and run the notebook with JupyterLab.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

REPO_NAME = "Indonesia-Financial-Inclusion-Digital-Finance-Analytics"
REPO_URL = "https://github.com/rrrafaa/" + REPO_NAME + ".git"
ZIP_NAME = "IDN_2024_FINDEX_v02_M_CSV.zip"
CSV_NAME = "Findex_Microdata_2025_updateIndonesia.csv"
ZIP_SHA256 = "fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
CSV_SHA256 = "77a50e351c16946ad107c08494efe9e8972ba628168fa385b771695971710093"

IN_COLAB = "google.colab" in sys.modules

def locate_repo():
    locations = [Path.cwd(), *Path.cwd().parents]
    if IN_COLAB:
        locations.append(Path("/content") / REPO_NAME)
    for loc in locations:
        if (loc / "data" / "raw" / "README.md").exists():
            return loc
    if IN_COLAB:
        destination = Path("/content") / REPO_NAME
        if not destination.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(destination)],
                check=True
            )
        if (destination / "data" / "raw" / "README.md").exists():
            return destination
    raise FileNotFoundError(
        "Repository not found. Clone it and launch Jupyter from its root."
    )

ROOT = locate_repo()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

source = next(
    (p for p in [RAW_DIR / ZIP_NAME, Path("/content") / ZIP_NAME, Path.cwd() / ZIP_NAME] if p.exists()),
    None
)
if source is None and IN_COLAB:
    from google.colab import files
    print("Upload the official ZIP file you downloaded privately from World Bank.")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError("Expected ZIP file: " + ZIP_NAME)
    source = RAW_DIR / ZIP_NAME
    source.write_bytes(uploaded[ZIP_NAME])

if source is None:
    raise FileNotFoundError(
        "Put the original " + ZIP_NAME + " in the local data/raw/ folder."
    )

def sha256_bytes(content):
    return hashlib.sha256(content).hexdigest()

raw_zip = source.read_bytes()
assert sha256_bytes(raw_zip) == ZIP_SHA256, (
    "ZIP SHA-256 mismatch: wrong dataset/version or corrupted ZIP."
)
with zipfile.ZipFile(io.BytesIO(raw_zip)) as z:
    if CSV_NAME not in z.namelist():
        raise FileNotFoundError("CSV not found in the verified ZIP: " + CSV_NAME)
    raw_csv = z.read(CSV_NAME)
assert sha256_bytes(raw_csv) == CSV_SHA256, "Unpacked CSV checksum mismatch."

df_raw = pd.read_csv(io.BytesIO(raw_csv), low_memory=False)
assert df_raw.shape == (1068, 199), f"Unexpected shape: {df_raw.shape}"
assert set(df_raw["year"].dropna().unique().tolist()) == {2024}

print("Source file: verified SHA-256 checksums")
print("Survey: Indonesia, 2024")
print("Rows:", len(df_raw), "| Columns:", df_raw.shape[1])
print("Working directory:", ROOT)
print("IMPORTANT: Raw and respondent-level files stay private.")

## 2. Validate source integrity, codes and survey routing

This stage must **fail loudly** if the source structure differs. Missing education and skipped barrier questions are not automatically zero-filled. Any failed assertion requires investigation rather than silently continuing.

In [ ]:
CORE = [
    "wpid_random", "year", "wgt", "female", "age", "educ",
    "inc_q", "urbanicity", "account", "account_fin", "account_mob",
    "anydigpayment", "internet_use", "saved", "borrowed"
]
BARRIERS = ["fin11a", "fin11b", "fin11c", "fin11d", "fin11e", "fin11f"]
missing_cols = sorted(set(CORE + BARRIERS) - set(df_raw.columns))
assert not missing_cols, f"Required columns absent: {missing_cols}"

def allowed(column, codes):
    values = set(df_raw[column].dropna().unique())
    assert values <= set(codes), (
        f"Unexpected answer code(s) in {column}: {values-set(codes)}"
    )

assert df_raw["wpid_random"].notna().all()
assert df_raw["wpid_random"].is_unique, "Duplicate respondent IDs: inspect first."
assert not df_raw.duplicated().any(), "Duplicate records found: inspect first."
assert np.isfinite(df_raw["wgt"]).all() and df_raw["wgt"].gt(0).all()
assert df_raw["age"].between(15, 120).all(), "Unexpected age range."

for name, accepted in {
    "female": [1, 2],
    "educ": [1, 2, 3],
    "inc_q": [1, 2, 3, 4, 5],
    "urbanicity": [1, 2]
}.items():
    allowed(name, accepted)

INDICATORS = {
    "account": "Owns an account",
    "account_fin": "Financial institution account",
    "account_mob": "Mobile money account",
    "anydigpayment": "Digital payment use",
    "internet_use": "Internet use",
    "saved": "Saved money",
    "borrowed": "Borrowed money",
}
for name in INDICATORS:
    allowed(name, [0, 1])

for name in BARRIERS:
    allowed(name, [1, 2, 3, 4])

eligible = df_raw["account_fin"].eq(0)
assert int(eligible.sum()) == 504, "Unexpected barrier eligibility count."
assert df_raw.loc[eligible, BARRIERS].notna().all().all(), (
    "Eligible respondents have missing barrier answers; inspect routing."
)
assert df_raw.loc[~eligible, BARRIERS].isna().all().all(), (
    "Ineligible respondents have barrier answers; inspect routing."
)
assert (df_raw["account"] >= df_raw["account_fin"]).all()
assert (df_raw["account"] >= df_raw["account_mob"]).all()

print("Data integrity passed.")
print("Missing education:", int(df_raw["educ"].isna().sum()))
print("Eligible for financial institution barriers:", int(eligible.sum()))
print("Ineligible (correctly skipped):", int((~eligible).sum()))
print("No missing values have been replaced with zero.")

## 3. Create one shared, canonical codebook and demographic labels

**Data dictionary policy:** Keep the numeric original variables for reproducibility. Add readable segment labels, while preserving true missing responses as missing. The 2024 `fin11d/e/f` mapping below is verified against the World Bank 2024 codebook (NOT the earlier Notebook 02 mapping).

In [ ]:
GENDER = {1: "Female", 2: "Male"}
EDUCATION = {1: "Primary or less", 2: "Secondary", 3: "Tertiary or more"}
INCOME = {1: "Q1 - lowest", 2: "Q2", 3: "Q3", 4: "Q4", 5: "Q5 - highest"}
RESIDENCE = {1: "Rural", 2: "Urban"}
INTERNET = {0: "Non-internet user", 1: "Internet user"}

# Verified 2024 labels — the mapping in the original Notebook 02 is inaccurate.
BARRIER_LABELS = {
    "fin11a": "Financial institutions are too far away",
    "fin11b": "Financial service fees are too expensive",
    "fin11c": "Lack of necessary documentation",
    "fin11d": "Not enough money to open an account",
    "fin11e": "Another family member has an account",
    "fin11f": "Lack of trust in financial institutions",
}

SELECTED_COLUMNS = [
    "year", "wgt", "female", "age", "educ", "inc_q", "urbanicity"
] + list(INDICATORS) + BARRIERS
# Intentionally exclude source respondent identifier from clean exported data.
assert "wpid_random" not in SELECTED_COLUMNS
clean = df_raw[SELECTED_COLUMNS].copy()

clean["gender"] = clean["female"].map(GENDER)
clean["education"] = clean["educ"].map(EDUCATION)
clean["income_quintile"] = clean["inc_q"].map(INCOME)
clean["residence"] = clean["urbanicity"].map(RESIDENCE)
clean["internet_group"] = clean["internet_use"].map(INTERNET)
clean["age_group"] = pd.cut(
    clean["age"],
    bins=[14, 24, 34, 44, 54, 120],
    labels=["15-24", "25-34", "35-44", "45-54", "55+"],
    include_lowest=True
).astype("string")

assert len(clean) == 1068
assert clean["education"].isna().sum() == df_raw["educ"].isna().sum()
assert clean[["gender", "income_quintile", "residence", "age_group"]].notna().all().all()
assert clean["wgt"].gt(0).all()
print("Integrated private respondent-level data prepared.")
print("Columns:", len(clean.columns), "| Rows:", len(clean))
print("Missing education kept as NA:", clean["education"].isna().sum())
print("Respondent identifiers NOT exported.")

## 4. Define consistent weighted measures for both analysts

A valid binary indicator has response 0/1. Weighted percentage = **100 × sum(wgt × indicator) / sum(wgt)** among valid responses in the relevant group. Do not take a simple mean of subgroup percentages. Both the numerator and denominator are exported so Looker Studio can aggregate compatible rows correctly.

In [ ]:
def weighted_binary_rate(frame, indicator):
    subset = frame.loc[
        frame[indicator].isin([0, 1]) & frame["wgt"].gt(0),
        [indicator, "wgt"]
    ]
    if subset.empty:
        raise ValueError(f"No valid responses for {indicator}")
    return {
        "numerator_wgt": float((subset[indicator] * subset["wgt"]).sum()),
        "denominator_wgt": float(subset["wgt"].sum()),
        "n_valid": int(len(subset)),
        "n_yes": int(subset[indicator].eq(1).sum()),
    }

def as_pct(numerator, denominator):
    if denominator <= 0:
        raise ValueError("Denominator must be positive")
    return 100 * numerator / denominator

DIMENSION_ORDER = {
    "gender": ["Female", "Male"],
    "income_quintile": ["Q1 - lowest", "Q2", "Q3", "Q4", "Q5 - highest"],
    "education": ["Primary or less", "Secondary", "Tertiary or more"],
    "age_group": ["15-24", "25-34", "35-44", "45-54", "55+"],
    "residence": ["Rural", "Urban"],
    "internet_group": ["Non-internet user", "Internet user"],
}

def kpi_row(indicator, dimension, category, sort_order, subset):
    result = weighted_binary_rate(subset, indicator)
    result.update({
        "metric": indicator,
        "metric_label": INDICATORS[indicator],
        "dimension": dimension,
        "category": category,
        "sort_order": sort_order,
        "rate_pct": as_pct(result["numerator_wgt"], result["denominator_wgt"])
    })
    return result

rows = []
for indicator in INDICATORS:
    rows.append(kpi_row(indicator, "national", "Indonesia", 0, clean))
    for dimension, labels in DIMENSION_ORDER.items():
        for sort_order, category in enumerate(labels, 1):
            group = clean.loc[clean[dimension].eq(category)]
            if group.empty:
                continue
            result = kpi_row(indicator, dimension, category, sort_order, group)
            # Conservative dashboard safeguard: never publish a cell with <30 valid rows.
            if result["n_valid"] >= 30:
                rows.append(result)

kpi_summary = pd.DataFrame(rows)
kpi_summary = kpi_summary[[
    "metric", "metric_label", "dimension", "category", "sort_order",
    "numerator_wgt", "denominator_wgt", "rate_pct", "n_valid", "n_yes"
]].sort_values(["metric", "dimension", "sort_order"]).reset_index(drop=True)
assert kpi_summary["rate_pct"].between(0, 100).all()
assert kpi_summary["n_valid"].ge(30).all()
assert not kpi_summary["category"].astype(str).str.lower().eq("nan").any()
print("Validated aggregate KPI rows:", len(kpi_summary))
display(kpi_summary.loc[kpi_summary["dimension"].eq("national"),
                    ["metric", "rate_pct", "n_valid"]].round(3))

## 5. Normalize barrier response codes and calculate barriers correctly

Barriers are **multi-response questions** asked only of respondents without a financial-institution account (`account_fin == 0`). Each question has codes **1=Yes, 2=No, 3=Don't know, 4=Refused**; only 1 and 2 enter the denominator. Empty fields for ineligible respondents are skip logic, not No.

In [ ]:
barrier_source = clean.loc[clean["account_fin"].eq(0), ["wgt"] + BARRIERS].copy()
barrier_long = barrier_source.melt(
    id_vars="wgt", value_vars=BARRIERS,
    var_name="barrier_code", value_name="answer_code"
)
barrier_long = barrier_long.loc[
    barrier_long["answer_code"].isin([1, 2])
].copy()
barrier_long["yes_weight"] = np.where(
    barrier_long["answer_code"].eq(1), barrier_long["wgt"], 0.0
)

barrier_summary = (
    barrier_long.groupby("barrier_code", as_index=False)
    .agg(
        numerator_wgt=("yes_weight", "sum"),
        denominator_wgt=("wgt", "sum"),
        n_valid=("answer_code", "size"),
        n_yes=("answer_code", lambda s: int(s.eq(1).sum()))
    )
)
barrier_summary["rate_pct"] = (
    100 * barrier_summary["numerator_wgt"] / barrier_summary["denominator_wgt"]
)
barrier_summary["barrier_label"] = barrier_summary["barrier_code"].map(BARRIER_LABELS)
barrier_summary = barrier_summary.sort_values(
    "rate_pct", ascending=False
).reset_index(drop=True)
barrier_summary.insert(0, "rank", np.arange(1, len(barrier_summary) + 1))
barrier_summary = barrier_summary[[
    "rank", "barrier_code", "barrier_label", "numerator_wgt",
    "denominator_wgt", "rate_pct", "n_valid", "n_yes"
]]

assert len(barrier_summary) == 6
assert set(barrier_summary["barrier_code"]) == set(BARRIERS)
assert barrier_summary["rate_pct"].between(0, 100).all()
assert barrier_summary["n_valid"].ge(30).all()
print("Barrier denominators correctly use Yes/No answers among eligible respondents.")
display(barrier_summary[["rank", "barrier_code", "barrier_label",
                         "rate_pct", "n_valid"]].round(3))

## 6. Cross-check against EDA Notebooks 01, 02 and 04

A tolerance of **0.15 percentage points** is allowed only to accommodate rounding in reported benchmarks. A mismatch makes execution stop; it must be investigated before preparing a dashboard.

In [ ]:
EXPECTED_NATIONAL = {
    "account": 56.33,
    "account_fin": 52.39,
    "account_mob": 22.01,
    "anydigpayment": 42.69,
    "internet_use": 70.5,
    "saved": 52.99,
    "borrowed": 47.64,
}
checks = []
def record(check, actual, expected, passed, note=""):
    checks.append({
        "check": check, "actual": actual, "expected": expected,
        "status": "PASS" if passed else "FAIL", "note": note
    })

record("input_shape", str(df_raw.shape), "(1068, 199)",
       df_raw.shape == (1068, 199), "Matched Notebook 01")
record("unique_respondents", bool(df_raw["wpid_random"].is_unique), True,
       bool(df_raw["wpid_random"].is_unique))
record("all_weights_positive", bool(df_raw["wgt"].gt(0).all()), True,
       bool(df_raw["wgt"].gt(0).all()))
record("education_missing", int(df_raw["educ"].isna().sum()), 1,
       int(df_raw["educ"].isna().sum()) == 1)
record("barrier_eligible_count", int(eligible.sum()), 504,
       int(eligible.sum()) == 504)
record("barriers_count", len(barrier_summary), 6, len(barrier_summary) == 6)

national = kpi_summary.loc[kpi_summary["dimension"].eq("national")].set_index("metric")
for metric, target in EXPECTED_NATIONAL.items():
    actual = float(national.at[metric, "rate_pct"])
    record(f"weighted_{metric}_pct", round(actual, 4), target,
           abs(actual - target) <= 0.15, "Rounded benchmark from EDA 01/02/04")

assert barrier_summary.iloc[0]["barrier_code"] == "fin11d", (
    "Unexpected leading financial barrier; inspect data and mapping."
)
record("top_barrier_code", str(barrier_summary.iloc[0]["barrier_code"]),
       "fin11d", True, "Correct 2024 meaning: Not enough money")

quality_checks = pd.DataFrame(checks)
display(quality_checks)
assert quality_checks["status"].eq("PASS").all(), (
    "One or more integration checks failed. Do not export dashboard tables."
)
print("ALL INTEGRATION CHECKS PASSED.")

## 7. Export data, documentation and reproducibility manifest

All exports go to `data/processed/` (excluded by Git). `integrated_respondents_private.csv` is still restricted microdata even without the original respondent identifier. **Do not publish it.** Use only the validated, aggregated CSVs for your Looker Studio source.

In [ ]:
# Private, person-level output: never commit or publish.
private_path = OUT_DIR / "integrated_respondents_private.csv"
clean.to_csv(private_path, index=False)

# Aggregates intended for review before Looker Studio upload.
kpi_path = OUT_DIR / "dashboard_kpi_aggregates.csv"
barrier_path = OUT_DIR / "dashboard_barriers_aggregates.csv"
qa_path = OUT_DIR / "integration_quality_checks.csv"
dictionary_path = OUT_DIR / "integrated_data_dictionary.csv"
manifest_path = OUT_DIR / "integration_manifest.json"

kpi_summary.to_csv(kpi_path, index=False, float_format="%.8f")
barrier_summary.to_csv(barrier_path, index=False, float_format="%.8f")
quality_checks.to_csv(qa_path, index=False)

dictionary_rows = [
    {"variable": v, "label": INDICATORS[v], "valid_codes": "0=No, 1=Yes",
     "eligible_population": "Survey respondents with valid answer",
     "weighted_denominator": "sum(wgt) over valid 0/1 respondents"}
    for v in INDICATORS
]
dictionary_rows += [
    {"variable": v, "label": BARRIER_LABELS[v],
     "valid_codes": "1=Yes, 2=No; 3=DK, 4=Refused are excluded",
     "eligible_population": "account_fin == 0",
     "weighted_denominator": "sum(wgt) over eligible valid 1/2 respondents"}
    for v in BARRIERS
]
dictionary_rows += [
    {"variable": "wgt", "label": "Survey weight", "valid_codes": "positive numeric",
     "eligible_population": "All respondents",
     "weighted_denominator": "Used as weight; valid sum(wgt) is denominator"},
    {"variable": "educ", "label": "Education code", "valid_codes": "1, 2, 3, or missing",
     "eligible_population": "All respondents",
     "weighted_denominator": "Missing education excluded from education segment only"}
]
pd.DataFrame(dictionary_rows).to_csv(dictionary_path, index=False)

manifest = {
    "project": REPO_NAME,
    "survey_country": "Indonesia",
    "survey_year": 2024,
    "source_zip": ZIP_NAME,
    "source_zip_sha256": ZIP_SHA256,
    "source_csv_sha256": CSV_SHA256,
    "raw_shape": list(df_raw.shape),
    "clean_shape": list(clean.shape),
    "expected_barrier_eligible_n": int(eligible.sum()),
    "missing_education_n": int(df_raw["educ"].isna().sum()),
    "qa_passed": bool(quality_checks["status"].eq("PASS").all()),
    "barrier_definition_source": "https://microdata.worldbank.org/catalog/7917/data-dictionary/F1",
    "private_data_is_not_redistributable": True,
    "dashboard_exports": [kpi_path.name, barrier_path.name]
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Files created in private Colab/local runtime:")
for target in [private_path, kpi_path, barrier_path, qa_path, dictionary_path, manifest_path]:
    print(" -", target.name)
print("QA status: PASS | Ready for analyst review (not yet an approved dashboard).")

## 8. Optional persistence: copy outputs to your PRIVATE Google Drive (Colab only)

By default, Colab runtime storage can disappear on disconnect. Set `SAVE_TO_PRIVATE_DRIVE = True` to save the generated files in a private `MyDrive/Financial_Inclusion_Private/processed/` directory. **Never set a public Anyone-with-the-link sharing permission for respondent-level data.** This is optional and does not publish anything to GitHub.

In [ ]:
SAVE_TO_PRIVATE_DRIVE = False

if SAVE_TO_PRIVATE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mount is only supported in Colab here.")
    from google.colab import drive
    drive.mount("/content/drive")
    private_drive_folder = Path("/content/drive/MyDrive/Financial_Inclusion_Private/processed")
    private_drive_folder.mkdir(parents=True, exist_ok=True)
    for file_to_save in [private_path, kpi_path, barrier_path, qa_path, dictionary_path, manifest_path]:
        shutil.copy2(file_to_save, private_drive_folder / file_to_save.name)
    print("Saved privately to:", private_drive_folder)
else:
    print("Private Drive copy disabled. Download/copy outputs before closing Colab.")

## 9. Integration handoff — what must happen before Looker Studio

**Notebook 03 is a validation/cleaning pipeline**, not a published dashboard. Two required follow-ups:

1. **Person 2:** Fix the incorrect 2024 `fin11d/e/f` labels in Notebook 02 and rerun any affected barrier chart/table. Person 2's original notebook contains additional exploratory work; preserve its history, do not overwrite their analysis blindly.
2. **Person 1:** Once Notebook 03 finishes with all checks passing, rerun Notebook 04 with the agreed demographic mappings and verify rates against `dashboard_kpi_aggregates.csv`. The current Notebook 04 independently reloads the raw ZIP; it does **not** automatically consume this Notebook 03 output.
3. **Both:** Confirm that the national rates, segment labels, denominators and source ZIP SHA match. Review missing education and subgroup uncertainty. Decide which approved aggregate tables will be available to Looker Studio.
4. **Dashboard build:** Use `dashboard_kpi_aggregates.csv` for Page 1–3 and `dashboard_barriers_aggregates.csv` for Page 4; filter by `dimension` and `metric`. A single-dimension pre-aggregated table **does not support arbitrary simultaneous demographic cross-filters** (e.g. selecting both Female and Q5) without additional joint aggregations. Plan these in Notebook 06 if needed.

### Completion checklist
- [ ] Source SHA-256 checks PASS.
- [ ] All categorical codes, missingness and barrier skip logic validated.
- [ ] 2024 World Bank `fin11d/e/f` mapping is correct.
- [ ] All integration QA checks PASS.
- [ ] Review aggregate tables and copy them to a private, persistent location.
- [ ] Person 2 reviews and corrects Notebook 02.
- [ ] Person 1 reruns Notebook 04 against the integrated definitions.
- [ ] Only then: build the Looker Studio dashboard and document limitations.
